In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Satıcılar

🎯 Amacımız, diğerlerine göre tekrarlı olarak düşük performans gösteren satıcıları bulmak ve nedenini anlamaktır.  
Bu, Olist'in gelecekte kâr marjını artırmaya yönelik önerilerimizi şekillendirmemize yardımcı olacaktır.

## 1 - `olist/seller.py`  

`order.py` ile benzer bir süreçte, `Seller` sınıfı ve `Seller().get_training_data` yöntemi sunuldu. Tek doldurmanız gereken: `get_review_score()`.

❓ **Aşağıya yeni sınıfınızı import edin ve eğitim veri çerçevenizi inceleyin!**

In [ ]:
from olist.seller import Seller

sellers = Seller().get_training_data()
sellers.head()

🤔 Her satıcı için hesaplanması gereken son şey:
- `share_of_five_stars`, `share_of_one_stars`
- (ortalama) `review_score`

❓ **Son metod: `get_review_score()`'u implemente edin.** (Çözüm `olist/seller.py` içine yazıldı.)

In [ ]:
# Implemented in olist/seller.py — quick look at the seller-grain review frame
Seller().get_review_score().head()

🧪 Kodunuzu aşağıda test edin

In [ ]:
from nbresult import ChallengeResult

tmp = Seller().get_training_data()
result = ChallengeResult('seller',
    shape = tmp.shape,
    median = tmp.review_score.median(),
    columns = tmp.columns
)
result.write()
print(result.check())

⚠️ `olist` reposundaki `seller.py` dosyasına yaptığınız kod değişikliklerini commit etmeyi unutmayın!

## 2 - Satıcıları İnceleme

### (2.1) Görselleştirmeler

👉 `sellers` için özet istatistiklere bakın. Satıcı başına siparişlerin medyanı nedir ❓

In [ ]:
sellers.describe()

👉 Satıcı başına sipariş medyanı:

In [ ]:
print(f"Median orders per seller: {sellers['n_orders'].median()}")

👉 Her sayısal değişkenin dağılımı (kod verildi, çalıştırın). Aykırı değer? Sipariş dağılımı nasıl?

In [ ]:
plt.figure(figsize=(15,11))
for (i, col) in enumerate(sellers.describe().columns):#["wait_time", "delay_to_carrier", "avg_review_score", "n_orders", "quantity", "price"]):
    plt.subplot(3,4,i+1)
    sns.histplot(sellers[col], kde=False, stat='density', discrete=[True,None][col in ['share_of_one_stars','share_of_five_stars','sales']]);

💡 Çok düşük puanlı bir grup satıcı öne çıkıyor! `plotly` ile inceleyelim:

In [ ]:
import plotly.express as px
fig = px.scatter(data_frame = sellers[sellers['review_score'] < 4],
    x="wait_time",
    y="delay_to_carrier",
    size="sales",
    color="review_score",
    size_max = 60,
    opacity = 0.5
)
fig.show()

En kötü satıcıları bulmak için `x`, `y`, `color`, `size` değiştirebilirsiniz.

### (2.2) `review_score`'u OLS ile modelleme

💡 Çok değişkenli OLS ile çeşitli özelliklerin `review_score` üzerindeki etkisini modelleyelim. Önce `standardize` ile özellikleri standardize edin.

In [ ]:
def standardize(df, features):
    """Standardize specified numerical features in a DataFrame using z-score."""
    df_standardized = df.copy()
    mu = df[features].mean()
    sigma = df[features].std()
    df_standardized[features] = (df[features] - mu) / sigma
    return df_standardized

In [ ]:
# Continuous numeric features (NO share_of_*_stars: derived from review_score -> leakage)
features = ['wait_time', 'delay_to_carrier', 'months_on_olist',
            'n_orders', 'quantity', 'quantity_per_order', 'sales']

sellers_standardized = standardize(sellers, features)
sellers_standardized[features].describe().round(2)   # mean ~0, std ~1

👉 Sonraki adımda bir OLS modeli oluşturun ve fit edin.

In [ ]:
model = smf.ols(formula=f"review_score ~ {'+ '.join(features)}", data=sellers_standardized).fit()

❓ En etkili özellikler hangileri? 👉 Sıralanmış katsayılarla bir 📊 `bar_plot` çizin.

In [ ]:
coefs = model.params.drop('Intercept').sort_values()
coefs.plot(kind='barh', figsize=(8,5), color='steelblue')
plt.title('Standardized coefficients on seller review_score')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

👉 Modelin performansını (`R-squared`) ve `residuals`'ı inceleyin

In [ ]:
print(f"R-squared: {model.rsquared:.4f}")
print(f"F p-value: {model.f_pvalue:.2e}")

predicted_review_score = model.predict(sellers_standardized)
residuals = sellers_standardized['review_score'] - predicted_review_score
print(f"Mean of residuals: {residuals.mean():.2e}")
rmse = np.sqrt((residuals ** 2).mean())
print(f"RMSE: {rmse:.4f}")

👉 Gerçek ve tahmin edilen puanları aynı grafikte karşılaştırın.

In [ ]:
sns.histplot(sellers['review_score'], bins=40, color='steelblue', label='actual', stat='density')
sns.histplot(predicted_review_score, bins=40, color='orange', label='predicted', stat='density')
plt.legend()
plt.title('Actual vs predicted seller review_score')
plt.show()

👉 Artıkları (residuals) görselleştirin

In [ ]:
sns.histplot(residuals, bins=40)
plt.title('Distribution of residuals')
plt.xlabel('residual (actual - predicted)')
plt.show()

### (2.3) Analize `seller_state` bilgisini ekleyin

❓ Sadece `seller_state`'e karşı `review_score` regresyonu. `return_significative_coef` ile anlamlı eyaletleri bulun. En iyi eyaletler?

💡 Kategorik için formülde `C(seller_state)` kullanın.

In [ ]:
from olist.utils import return_significative_coef

model_state = smf.ols(formula='review_score ~ C(seller_state)', data=sellers).fit()
return_significative_coef(model_state)

❓ **`seller_state` etkisini izole edin: sürekli özellikleri ekleyin; `seller_state` artık anlamlı olmayana kadar.**

In [ ]:
# Add wait_time alongside the state dummies
model_state2 = smf.ols(
    formula='review_score ~ C(seller_state) + wait_time',
    data=sellers).fit()
return_significative_coef(model_state2)

In [ ]:
# How many state dummies remain significant once wait_time is controlled for?
sig = return_significative_coef(model_state2)
state_rows = sig[sig['variable'].str.contains('seller_state')]
print(f"Significant seller_state dummies after adding wait_time: {len(state_rows)}")
state_rows

☝️ `wait_time` eklendikten sonra 22 `is_seller_state_xx` dummy'sinin hiçbiri anlamlı kalmadı.

Küçük veri setimizde (çoğu eyalette az satıcı):
- "Bazı eyaletler `wait_time` dışı nedenlerle doğal olarak daha iyi" sonucuna **varamayız**.
- Yani "`seller_state`'in `wait_time` dışında etkisi yok" hipotezini reddedemeyiz.

> 📝 **Yorum (Poi):** Eyalet tek başına anlamlı görünüyordu; ama `wait_time` modele girince eyalet anlamlılığı kayboldu. 
> Demek ki "iyi eyalet" aslında "teslimatı hızlı eyalet" demekmiş — `seller_state`, `wait_time` için bir **vekil (proxy)** rolü oynuyordu. 
> Confounding'in bir başka örneği: ham ilişki gerçek bir nedensel etki değil, altta yatan teslimat hızının yansımasıydı.

🏁 Tebrikler!

💾 Commit ve push: `sellers.ipynb` + `seller.py`